# VWD Step 2 — YOLOX + Pose + IOD

This notebook contains only the Step 2 inference pipeline:
- YOLOX detections
- Pose-based torso + hands region
- IOD overlap filtering
- Optional visual / JSON output
- Excel output
- `weapon` / `no weapon` classification

To switch pose models, only change `POSE_WEIGHTS_PATH` in the config cell.

In [89]:
import sys
import os

print("Python executable:", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV"))

Python executable: /opt/anaconda3/envs/vwd_result_analyser/bin/python
Conda environment: vwd_result_analyser


In [90]:
import os
import re
import json
from pathlib import Path
from typing import List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
import onnxruntime
import torch
from ultralytics import YOLO

from yolox.utils import preproc as preprocess, mkdir, multiclass_nms, demo_postprocess
from yolox.visualize import vis as vis_yolox


In [91]:
# =========================
# CONFIG
# =========================

PATH_INPUT = "/Users/sushruthabhat/Documents/download_data_insight/monumental_fps/vwd_frames"
OUTPUT_DIR = "/Users/sushruthabhat/Documents/download_data_insight/monumental_fps/monumental_vwd_frames_vw3"
OUTPUT_EXCEL = "/Users/sushruthabhat/Documents/download_data_insight/monumental_fps/monumental_vwd_frames_vw3.xlsx"

YOLOX_ONNX_PATH = "/Users/sushruthabhat/Documents/vwd_result_analyser/yolox_models/vw3-7903-yxl-6.onnx"

# Change only this path if you want to switch pose models:
# e.g. "/path/to/yolo11m-pose.pt"
# or   "/path/to/yolov8m-pose.pt"
POSE_WEIGHTS_PATH = "/Users/sushruthabhat/Documents/vwd_result_analyser/yolox_models/yolo11m-pose.pt"

YOLOX_SCORE_THR_LIST = [0.70]

INPUT_SHAPE = (640, 640)
YOLOX_NMS_THR = 0.45
POSE_CONF_THR = 0.7
AREA_FRAC_THRESHOLD = 0.06

OVERLAP_METRIC = "iod_det"
OVERLAP_THRESHOLD = 0.6
KEEP_ALL_IF_NO_TORSO = False

SAVE_VISUALS = True
SAVE_JSON = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [92]:
# =========================
# CLASSES + WEAPON STATUS
# =========================

# vw3
COCO_CLASSES = (
    "pistol_in_hand",
    "heavy_weapon",
    "pistol_in_holster",
    "no_weapon",
    "civilian",
    "policeman"
)

# vw5
# COCO_CLASSES = (
#     "pistol_in_hand",
#     "pistol_in_holster",
#     "policeman",
#     "civilian", "no_weapon", "knife", "heavy_weapon"
# )


# vw6
# COCO_CLASSES = (
#     "pistol_in_hand",
#     "pistol_in_holster",
#     "policeman", "knife", "heavy_weapon"
# )

# Same status logic from the original code:
# only civilian / no_weapon => no weapon
# anything else => weapon
ALLOWED_NO_WEAPON_CLASSES = {"civilian", "no_weapon"}

IDX = {
    "nose": 0, "leye": 1, "reye": 2, "lear": 3, "rear": 4,
    "lsho": 5, "rsho": 6, "lelb": 7, "relb": 8, "lwri": 9, "rwri": 10,
    "lhip": 11, "rhip": 12, "lkne": 13, "rkne": 14, "lank": 15, "rank": 16,
}

HAND_PAD_X_FRAC = 0.1
HAND_PAD_Y_FRAC = 0.04


def infer_weapon_status(predicted_classes: str) -> str:
    if pd.isna(predicted_classes) or str(predicted_classes).strip() == "":
        return "no weapon"

    parts = [
        p.strip().lower().replace(" ", "_")
        for p in str(predicted_classes).split(",")
        if p.strip()
    ]

    return (
        "no weapon"
        if all(p in ALLOWED_NO_WEAPON_CLASSES for p in parts)
        else "weapon"
    )


In [93]:
# =========================
# MODEL / POSE HELPERS
# =========================

def create_onnx_session(model_path: str) -> onnxruntime.InferenceSession:
    providers = onnxruntime.get_available_providers()

    if "CUDAExecutionProvider" in providers:
        chosen = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        dev = "cuda"
    else:
        chosen = ["CPUExecutionProvider"]
        dev = "cpu"

    sess = onnxruntime.InferenceSession(model_path, providers=chosen)
    print(f"[INFO] YOLOX ONNXRuntime providers: {chosen} (device={dev})")
    return sess


def _kp_or_none(kps, idx):
    x, y = float(kps[idx][0]), float(kps[idx][1])
    if int(x) == 0 and int(y) == 0:
        return None
    return (x, y)


def compute_torso_bbox(
    keypoints: Optional[List[Tuple[float, float]]],
    person_bbox_xyxy: Tuple[int, int, int, int],
    img_w: int,
    img_h: int,
    pad_frac: float = 0.03,
    up_extra_frac: float = 0.03,
    below_knee_frac: float = 0.02,
    max_torso_rel_height: float = 0.70,
) -> Tuple[int, int, int, int]:

    bx1, by1, bx2, by2 = person_bbox_xyxy
    bw = max(1, bx2 - bx1)
    bh = max(1, by2 - by1)

    if not keypoints:
        pad_x = int(round(pad_frac * bw))
        h_cap = int(round(max_torso_rel_height * bh))
        x1 = max(0, bx1 + pad_x)
        x2 = min(img_w - 1, bx2 - pad_x)
        y_mid = (by1 + by2) // 2
        y1 = max(0, y_mid - h_cap // 2)
        y2 = min(img_h - 1, y1 + h_cap)
        return x1, y1, x2, y2

    pts = {name: _kp_or_none(keypoints, IDX[name]) for name in IDX}

    priority_top = ["lwri", "rwri", "lelb", "relb", "lsho", "rsho"]
    tops = [pts[n][1] for n in priority_top if pts.get(n) is not None]

    if tops:
        y_top = int(round(min(tops) - up_extra_frac * bh))
    else:
        any_kps = [p[1] for p in pts.values() if p is not None]
        y_top = int(round(min(any_kps))) if any_kps else int(round(by1 + 0.05 * bh))

    nose = pts.get("nose")
    if nose is not None:
        head_cut = int(round(nose[1] + 0.03 * bh))
        y_top = max(y_top, head_cut)

    knee_ys = [pts[n][1] for n in ["lkne", "rkne"] if pts.get(n) is not None]

    if knee_ys:
        y_bot = int(round(max(knee_ys) + below_knee_frac * bh))
    else:
        hip_ys = [pts[n][1] for n in ["lhip", "rhip"] if pts.get(n) is not None]
        if hip_ys:
            y_bot = int(round(max(hip_ys) + 0.40 * bh))
        else:
            y_bot = int(round(by1 + 0.65 * bh))

    if y_bot <= y_top:
        y_bot = y_top + max(10, int(0.25 * bh))

    x_candidates = [
        pts[n][0]
        for n in [
            "lsho", "rsho", "lelb", "relb", "lwri", "rwri",
            "lhip", "rhip", "lkne", "rkne"
        ]
        if pts.get(n) is not None
    ]

    if x_candidates:
        x1 = int(round(min(x_candidates)))
        x2 = int(round(max(x_candidates)))
    else:
        x1, x2 = bx1, bx2

    lwri = pts.get("lwri")
    rwri = pts.get("rwri")
    lelb = pts.get("lelb")
    relb = pts.get("relb")

    hand_xs = [p[0] for p in [lwri, rwri, lelb, relb] if p is not None]
    if hand_xs:
        hand_pad_x = int(round(HAND_PAD_X_FRAC * bw))
        x1 = min(x1, int(round(min(hand_xs))) - hand_pad_x)
        x2 = max(x2, int(round(max(hand_xs))) + hand_pad_x)

    hand_ys = [p[1] for p in [lwri, rwri, lelb, relb] if p is not None]
    if hand_ys:
        hand_pad_y = int(round(HAND_PAD_Y_FRAC * bh))
        min_hand_y = int(round(min(hand_ys))) - hand_pad_y
        max_hand_y = int(round(max(hand_ys))) + hand_pad_y
        y_top = min(y_top, min_hand_y)
        y_bot = max(y_bot, max_hand_y)

    pad_x = int(round(pad_frac * bw))
    x1 = max(0, x1 - pad_x)
    x2 = min(img_w - 1, x2 + pad_x)
    y_top = max(0, y_top)
    y_bot = min(img_h - 1, y_bot)

    max_h = int(round(max_torso_rel_height * bh))
    cur_h = y_bot - y_top

    if cur_h > max_h:
        center_y = (y_top + y_bot) // 2
        half = max_h // 2
        new_y1 = max(0, center_y - half)
        new_y2 = min(img_h - 1, new_y1 + max_h)

        if hand_ys:
            if (new_y1 <= min_hand_y) and (new_y2 >= max_hand_y):
                y_top, y_bot = new_y1, new_y2
        else:
            y_top, y_bot = new_y1, new_y2

    if x2 <= x1:
        x2 = min(img_w - 1, x1 + max(10, int(0.2 * bw)))

    if y_bot <= y_top:
        y_bot = min(img_h - 1, y_top + max(10, int(0.2 * bh)))

    return x1, y_top, x2, y_bot


def pick_best_person_forefront(result, img_wh, area_frac_thresh: float):
    w, h = img_wh
    img_area = float(w * h)

    if result.boxes is None:
        return None, None

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy().tolist()

    kps = (
        result.keypoints.xy.cpu().numpy()
        if result.keypoints is not None and result.keypoints.xy is not None
        else None
    )

    FRONT_BAND_MIN_BOTTOM_FRAC = 0.8
    BOTTOM_WEIGHT_EXP = 2.0
    WIDTH_WEIGHT_EXP = 1.0

    best_idx_gate = -1
    best_score_gate = -1.0
    best_idx_fallback = -1
    best_score_fallback = -1.0

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.tolist()

        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        area = bw * bh

        if area < area_frac_thresh * img_area:
            continue

        conf = float(confs[i])
        rel_area = area / img_area
        y_bottom_frac = max(0.0, min(1.0, y2 / h))
        width_frac = max(0.0, min(1.0, bw / w))

        score = conf * (
            rel_area
            * (1.0 + (y_bottom_frac ** BOTTOM_WEIGHT_EXP))
            * (0.5 + 0.5 * (width_frac ** WIDTH_WEIGHT_EXP))
        )

        if score > best_score_fallback:
            best_score_fallback = score
            best_idx_fallback = i

        if y_bottom_frac >= FRONT_BAND_MIN_BOTTOM_FRAC:
            if score > best_score_gate:
                best_score_gate = score
                best_idx_gate = i

    chosen = best_idx_gate if best_idx_gate >= 0 else best_idx_fallback

    if chosen < 0:
        return None, None

    x1, y1, x2, y2 = boxes[chosen].tolist()
    box_int = (
        int(round(x1)),
        int(round(y1)),
        int(round(x2)),
        int(round(y2)),
    )

    kp_list = None
    if kps is not None and chosen < len(kps):
        kp = kps[chosen]
        kp_list = [(float(x), float(y)) for (x, y) in kp]

    return box_int, kp_list


In [94]:
# =========================
# YOLOX + IOD LOGIC
# =========================

def run_yolox_image(session, frame, input_shape, score_thr, nms_thr):
    img, ratio = preprocess(frame, input_shape)

    ort_inputs = {
        session.get_inputs()[0].name: img[None, :, :, :]
    }

    output = session.run(None, ort_inputs)
    predictions = demo_postprocess(output[0], input_shape)[0]

    boxes = predictions[:, :4]
    scores = predictions[:, 4:5] * predictions[:, 5:]

    boxes_xyxy = np.ones_like(boxes)
    boxes_xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2.0
    boxes_xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2.0
    boxes_xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2.0
    boxes_xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2.0
    boxes_xyxy /= ratio

    dets = multiclass_nms(
        boxes_xyxy,
        scores,
        nms_thr=nms_thr,
        score_thr=score_thr,
    )

    if dets is None:
        return None

    return dets


def box_area_xyxy(b: Tuple[int, int, int, int]) -> int:
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


def intersection_area_xyxy(
    a: Tuple[int, int, int, int],
    b: Tuple[int, int, int, int],
) -> int:

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    return max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)


def overlap_xyxy(
    torso_xyxy: Tuple[int, int, int, int],
    det_xyxy: Tuple[int, int, int, int],
    metric: str = "iod_det",
) -> float:

    inter = intersection_area_xyxy(torso_xyxy, det_xyxy)

    if inter <= 0:
        return 0.0

    area_torso = box_area_xyxy(torso_xyxy)
    area_det = box_area_xyxy(det_xyxy)

    if area_torso == 0 or area_det == 0:
        return 0.0

    if metric == "iou":
        union = area_torso + area_det - inter
        return inter / union if union > 0 else 0.0

    elif metric == "iod_det":
        return inter / area_det

    elif metric == "iot_torso":
        return inter / area_torso

    elif metric == "iomin":
        return inter / min(area_torso, area_det)

    elif metric == "iomax":
        return inter / max(area_torso, area_det)

    raise ValueError(f"Unknown metric: {metric}")


def filter_dets_by_torso_overlap(
    dets: Optional[np.ndarray],
    torso_xyxy: Optional[Tuple[int, int, int, int]],
    thr: float,
    metric: str = "iod_det",
) -> Optional[np.ndarray]:

    if dets is None or len(dets) == 0:
        return None

    if torso_xyxy is None:
        return dets if KEEP_ALL_IF_NO_TORSO else None

    kept = []

    for row in dets:
        x1, y1, x2, y2, sc, cls_id = row.tolist()

        det_box = (
            int(x1),
            int(y1),
            int(x2),
            int(y2),
        )

        ov = overlap_xyxy(
            torso_xyxy,
            det_box,
            metric=metric,
        )

        cls_name = COCO_CLASSES[int(cls_id)]

        print(
            f"IOD MATCH | "
            f"class={cls_name} | "
            f"confidence={sc:.3f} | "
            f"{metric}={ov:.3f} ({ov * 100:.1f}%) | "
            f"threshold={thr:.2f} | "
            f"{'KEEP' if ov >= thr else 'REJECT'}"
        )

        if ov >= thr:
            kept.append(row)

    if not kept:
        return None

    return np.stack(kept, axis=0)


def draw_torso_box(img, xyxy, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = map(int, xyxy)

    cv2.rectangle(
        img,
        (x1, y1),
        (x2, y2),
        color,
        thickness,
    )

    cv2.putText(
        img,
        "torso+hands",
        (x1, max(0, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2,
        cv2.LINE_AA,
    )


In [95]:
# =========================
# EXCEL OUTPUT
# =========================

# Extract a UUID/GUID from the beginning of a filename.
# Works for names such as:
#   <guid>_visual-weapon-detection-frame.jpg
#   <guid>_entry-walk-image.jpg
#   <guid>_entry-walk-image.jpeg
# and does not depend on the suffix after the GUID.
GUID_RE = re.compile(
    r"^([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-"
    r"[0-9a-fA-F]{4}-[0-9a-fA-F]{12})(?=_|\.|#|$)"
)


def extract_guid(image_name: str) -> str:
    base_name = os.path.basename(str(image_name))
    match = GUID_RE.match(base_name)
    return match.group(1) if match else ""


def format_bbox_xyxy(box) -> str:
    if box is None:
        return ""

    x1, y1, x2, y2 = box
    return f"[{int(x1)} {int(y1)} {int(x2)} {int(y2)}]"


def log_row_for_excel(
    excel_rows: List[dict],
    image_name: str,
    dets_filtered: Optional[np.ndarray],
    torso_xyxy: Optional[Tuple[int, int, int, int]] = None,
):
    guid = extract_guid(image_name)
    torso_bbox = format_bbox_xyxy(torso_xyxy)

    if dets_filtered is None or len(dets_filtered) == 0:
        predicted_classes = ""

        excel_rows.append({
            "image_name": image_name,
            "guid": guid,
            "predicted_yolox_class": predicted_classes,
            "bbox_xyxy": "",
            "confidence": "",
            "torso_bbox_xyxy": torso_bbox,
            "iod": "",
            "vwd_prediction": infer_weapon_status(predicted_classes),
        })
        return

    classes = []
    bboxes = []
    confs = []
    iod_values = []

    for row in dets_filtered:
        x1, y1, x2, y2, sc, cls_id = row.tolist()

        classes.append(COCO_CLASSES[int(cls_id)])
        bboxes.append(
            f"[{int(x1)} {int(y1)} {int(x2)} {int(y2)}]"
        )

        # Keep raw YOLOX score without rounding.
        confs.append(repr(float(sc)))

        # One IoD value for each kept YOLOX detection, in the same order
        # as predicted_yolox_class / bbox_xyxy / confidence.
        if torso_xyxy is not None:
            det_box = (
                int(x1),
                int(y1),
                int(x2),
                int(y2),
            )
            ov = overlap_xyxy(
                torso_xyxy,
                det_box,
                metric=OVERLAP_METRIC,
            )
            iod_values.append(repr(float(ov)))
        else:
            iod_values.append("")

    predicted_classes = ",".join(classes)

    excel_rows.append({
        "image_name": image_name,
        "guid": guid,
        "predicted_yolox_class": predicted_classes,
        "bbox_xyxy": ",".join(bboxes),
        "confidence": ",".join(confs),
        "torso_bbox_xyxy": torso_bbox,
        "iod": ",".join(iod_values),
        "vwd_prediction": infer_weapon_status(predicted_classes),
    })


In [96]:
# =========================
# IMAGE PROCESSING
# =========================

def process_image_iod(
    image_path: str,
    yolox_sess,
    pose_model,
    out_dir: Path,
    json_store: dict,
    score_thr_yx: float,
):
    name = os.path.basename(image_path)

    frame = cv2.imread(image_path)
    if frame is None:
        print(f"[WARN] Could not read {image_path}")
        return None, None

    h, w = frame.shape[:2]

    dets = run_yolox_image(
        yolox_sess,
        frame,
        INPUT_SHAPE,
        score_thr_yx,
        YOLOX_NMS_THR,
    )

    result = pose_model(
        frame,
        conf=POSE_CONF_THR,
        verbose=False,
        device=DEVICE,
    )[0]

    person_box, keypoints = pick_best_person_forefront(
        result,
        (w, h),
        AREA_FRAC_THRESHOLD,
    )

    torso_xyxy = None

    if person_box is not None:
        tx1, ty1, tx2, ty2 = compute_torso_bbox(
            keypoints=keypoints,
            person_bbox_xyxy=person_box,
            img_w=w,
            img_h=h,
            pad_frac=0.03,
            up_extra_frac=0.03,
            below_knee_frac=0.02,
            max_torso_rel_height=0.70,
        )

        torso_xyxy = (tx1, ty1, tx2, ty2)

    dets_filtered = filter_dets_by_torso_overlap(
        dets,
        torso_xyxy,
        thr=OVERLAP_THRESHOLD,
        metric=OVERLAP_METRIC,
    )

    if torso_xyxy is not None:
        print(
            f"{name} | TORSO_XYXY: "
            f"[{torso_xyxy[0]}, {torso_xyxy[1]}, "
            f"{torso_xyxy[2]}, {torso_xyxy[3]}]"
        )
    else:
        print(f"{name} | TORSO_XYXY: person_not_found")

    if dets_filtered is not None:
        for row in dets_filtered:
            x1, y1, x2, y2, sc, cls_id = row.tolist()
            cls_name = COCO_CLASSES[int(cls_id)]

            print(
                f"{name} | "
                f"YOLOX_DET[{OVERLAP_METRIC}≥{OVERLAP_THRESHOLD:.2f}, "
                f"thr={score_thr_yx:.2f}]: "
                f"[{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}], "
                f"score={repr(float(sc))}, cls={cls_name}"
            )
    else:
        print(
            f"{name} | "
            f"YOLOX_DET[{OVERLAP_METRIC}≥{OVERLAP_THRESHOLD:.2f}, "
            f"thr={score_thr_yx:.2f}]: none"
        )

    if json_store is not None:
        entry = {}

        if torso_xyxy is not None:
            entry["torso_xyxy"] = [
                int(torso_xyxy[0]),
                int(torso_xyxy[1]),
                int(torso_xyxy[2]),
                int(torso_xyxy[3]),
            ]

        if dets_filtered is not None:
            det_list = []

            for row in dets_filtered:
                x1, y1, x2, y2, sc, cls_id = row.tolist()

                det_list.append({
                    "xyxy": [
                        int(x1),
                        int(y1),
                        int(x2),
                        int(y2),
                    ],
                    "score": float(sc),
                    "class": COCO_CLASSES[int(cls_id)],
                })

            entry["yolox_dets_overlap_filtered"] = det_list

        json_store[name] = entry

    if SAVE_VISUALS:
        canvas = frame.copy()

        if dets_filtered is not None:
            final_boxes = dets_filtered[:, :4]
            final_scores = dets_filtered[:, 4]
            final_cls_inds = dets_filtered[:, 5]

            canvas = vis_yolox(
                canvas,
                final_boxes,
                final_scores,
                final_cls_inds,
                conf=score_thr_yx,
                class_names=COCO_CLASSES,
            )

        if torso_xyxy is not None:
            draw_torso_box(
                canvas,
                torso_xyxy,
                (0, 255, 0),
                2,
            )

        out_dir.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_dir / name), canvas)

    return dets_filtered, torso_xyxy


In [97]:
# =========================
# VIDEO PROCESSING
# =========================

def process_video_iod(
    video_path: str,
    yolox_sess,
    pose_model,
    out_dir: Path,
    json_store: dict,
    score_thr_yx: float,
):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"[WARN] Could not open {video_path}")
        return {}

    video_name = os.path.basename(video_path)
    vid_json = {}
    out_writer = None

    if SAVE_VISUALS:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        Path(out_dir).mkdir(parents=True, exist_ok=True)

        out_path = str(Path(out_dir) / video_name)
        out_writer = cv2.VideoWriter(
            out_path,
            fourcc,
            fps,
            (w, h),
        )

    frame_id = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        dets = run_yolox_image(
            yolox_sess,
            frame,
            INPUT_SHAPE,
            score_thr_yx,
            YOLOX_NMS_THR,
        )

        h, w = frame.shape[:2]

        result = pose_model(
            frame,
            conf=POSE_CONF_THR,
            verbose=False,
            device=DEVICE,
        )[0]

        person_box, keypoints = pick_best_person_forefront(
            result,
            (w, h),
            AREA_FRAC_THRESHOLD,
        )

        torso_xyxy = None

        if person_box is not None:
            tx1, ty1, tx2, ty2 = compute_torso_bbox(
                keypoints=keypoints,
                person_bbox_xyxy=person_box,
                img_w=w,
                img_h=h,
                pad_frac=0.03,
                up_extra_frac=0.03,
                below_knee_frac=0.02,
                max_torso_rel_height=0.70,
            )

            torso_xyxy = (tx1, ty1, tx2, ty2)

        dets_filtered = filter_dets_by_torso_overlap(
            dets,
            torso_xyxy,
            thr=OVERLAP_THRESHOLD,
            metric=OVERLAP_METRIC,
        )

        if torso_xyxy is not None:
            print(
                f"{video_name}#frame_{frame_id} | "
                f"TORSO_XYXY: "
                f"[{torso_xyxy[0]}, {torso_xyxy[1]}, "
                f"{torso_xyxy[2]}, {torso_xyxy[3]}]"
            )
        else:
            print(
                f"{video_name}#frame_{frame_id} | "
                f"TORSO_XYXY: person_not_found"
            )

        if dets_filtered is not None:
            for row in dets_filtered:
                x1, y1, x2, y2, sc, cls_id = row.tolist()
                cls_name = COCO_CLASSES[int(cls_id)]

                print(
                    f"{video_name}#frame_{frame_id} | "
                    f"YOLOX_DET[{OVERLAP_METRIC}≥{OVERLAP_THRESHOLD:.2f}, "
                    f"thr={score_thr_yx:.2f}]: "
                    f"[{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}], "
                    f"score={repr(float(sc))}, cls={cls_name}"
                )
        else:
            print(
                f"{video_name}#frame_{frame_id} | "
                f"YOLOX_DET[{OVERLAP_METRIC}≥{OVERLAP_THRESHOLD:.2f}, "
                f"thr={score_thr_yx:.2f}]: none"
            )

        entry = {}

        if torso_xyxy is not None:
            entry["torso_xyxy"] = [
                int(torso_xyxy[0]),
                int(torso_xyxy[1]),
                int(torso_xyxy[2]),
                int(torso_xyxy[3]),
            ]

        if dets_filtered is not None:
            det_list = []

            for row in dets_filtered:
                x1, y1, x2, y2, sc, cls_id = row.tolist()

                det_list.append({
                    "xyxy": [
                        int(x1),
                        int(y1),
                        int(x2),
                        int(y2),
                    ],
                    "score": float(sc),
                    "class": COCO_CLASSES[int(cls_id)],
                })

            entry["yolox_dets_overlap_filtered"] = det_list

        vid_json[f"frame_{frame_id}"] = entry

        if SAVE_VISUALS and out_writer is not None:
            canvas = frame.copy()

            if dets_filtered is not None:
                final_boxes = dets_filtered[:, :4]
                final_scores = dets_filtered[:, 4]
                final_cls_inds = dets_filtered[:, 5]

                canvas = vis_yolox(
                    canvas,
                    final_boxes,
                    final_scores,
                    final_cls_inds,
                    conf=score_thr_yx,
                    class_names=COCO_CLASSES,
                )

            if torso_xyxy is not None:
                draw_torso_box(
                    canvas,
                    torso_xyxy,
                    (0, 255, 0),
                    2,
                )

            out_writer.write(canvas)

        frame_id += 1

    cap.release()

    if out_writer is not None:
        out_writer.release()

    return vid_json


In [98]:
# =========================
# RUN STEP 2
# =========================

def fmt_thr(thr: float) -> str:
    return f"{thr:.2f}".replace(".", "_")


def excel_path_with_suffix(base_excel: str, suffix: str) -> Path:
    p = Path(base_excel)
    return p.with_name(p.stem + f"_{suffix}" + p.suffix)


def run_step2():
    print(f"[INFO] Ultralytics Pose device: {DEVICE}")

    yolox_sess = create_onnx_session(YOLOX_ONNX_PATH)
    pose_model = YOLO(POSE_WEIGHTS_PATH).to(DEVICE)

    in_path = Path(PATH_INPUT)
    all_files = []

    if in_path.is_dir():
        files = [str(in_path / f) for f in os.listdir(in_path)]

        for f in sorted(files):
            f_low = f.lower()

            if f_low.endswith(
                (
                    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
                    ".mp4", ".avi", ".mov", ".mkv", ".wmv",
                )
            ):
                all_files.append(f)

    else:
        f_low = str(in_path).lower()

        if f_low.endswith(
            (
                ".jpg", ".jpeg", ".png", ".bmp", ".webp",
                ".mp4", ".avi", ".mov", ".mkv", ".wmv",
            )
        ):
            all_files.append(str(in_path))
        else:
            print(f"[ERROR] Unsupported file format: {in_path}")
            return

    for thr in YOLOX_SCORE_THR_LIST:
        tag = f"thr_{fmt_thr(thr)}"
        out_dir_thr = Path(f"{OUTPUT_DIR}_{tag}")
        mkdir(out_dir_thr)

        json_results = {}
        excel_rows: List[dict] = []

        print(
            f"\n========== Running YOLOX at "
            f"score threshold = {thr:.2f} ==========\n"
        )

        for f in all_files:
            f_low = f.lower()

            if f_low.endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ):
                dets_filtered, torso_xyxy = process_image_iod(
                    f,
                    yolox_sess,
                    pose_model,
                    out_dir_thr,
                    json_results,
                    score_thr_yx=thr,
                )

                image_name = os.path.basename(f)

                log_row_for_excel(
                    excel_rows,
                    image_name,
                    dets_filtered,
                    torso_xyxy,
                )

            elif f_low.endswith(
                (".mp4", ".avi", ".mov", ".mkv", ".wmv")
            ):
                vid_json = process_video_iod(
                    f,
                    yolox_sess,
                    pose_model,
                    out_dir_thr,
                    json_results,
                    score_thr_yx=thr,
                )

                video_name = os.path.basename(f)

                if not vid_json:
                    log_row_for_excel(
                        excel_rows,
                        video_name,
                        None,
                        None,
                    )

                else:
                    for frame_key, entry in vid_json.items():
                        dets_here = None

                        if (
                            entry
                            and "yolox_dets_overlap_filtered" in entry
                        ):
                            rows = []

                            for d in entry[
                                "yolox_dets_overlap_filtered"
                            ]:
                                x1, y1, x2, y2 = d["xyxy"]
                                sc = d["score"]
                                cls_name = d["class"]
                                cls_id = COCO_CLASSES.index(cls_name)

                                rows.append(
                                    [x1, y1, x2, y2, sc, cls_id]
                                )

                            if rows:
                                dets_here = np.array(
                                    rows,
                                    dtype=float,
                                )

                        torso_here = None
                        if entry and "torso_xyxy" in entry:
                            torso_here = tuple(entry["torso_xyxy"])

                        frame_tag = (
                            f"{video_name}#"
                            f"{frame_key.split('_')[-1]}"
                        )

                        log_row_for_excel(
                            excel_rows,
                            frame_tag,
                            dets_here,
                            torso_here,
                        )

                if vid_json:
                    json_results[video_name] = vid_json

        if SAVE_JSON and json_results:
            json_path = out_dir_thr / "results.json"

            with open(json_path, "w") as jf:
                json.dump(
                    json_results,
                    jf,
                    indent=4,
                )

            print(f"[INFO] JSON saved at {json_path}")

        out_xlsx = excel_path_with_suffix(
            OUTPUT_EXCEL,
            tag,
        )

        Path(out_xlsx).parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        df = pd.DataFrame(
            excel_rows,
            columns=[
                "image_name",
                "guid",
                "predicted_yolox_class",
                "bbox_xyxy",
                "confidence",
                "torso_bbox_xyxy",
                "iod",
                "vwd_prediction",
            ],
        )

        df.to_excel(
            out_xlsx,
            index=False,
        )

        print(
            f"[INFO] Excel saved at {out_xlsx} "
            f"with {len(df)} row(s)."
        )

    return df


In [99]:
# Run
results_df = run_step2()
results_df.head()


[INFO] Ultralytics Pose device: cpu
[INFO] YOLOX ONNXRuntime providers: ['CPUExecutionProvider'] (device=cpu)

========== Running YOLOX at score threshold = 0.70 ==========

IOD MATCH | class=pistol_in_hand | confidence=0.797 | iod_det=1.000 (100.0%) | threshold=0.60 | KEEP
0014a0ba-ac73-11f1-8c55-0242ac110014_visual-weapon-detection-frame.jpg | TORSO_XYXY: [38, 467, 509, 1066]
0014a0ba-ac73-11f1-8c55-0242ac110014_visual-weapon-detection-frame.jpg | YOLOX_DET[iod_det≥0.60, thr=0.70]: [403, 835, 483, 1049], score=0.7966657280921936, cls=pistol_in_hand
IOD MATCH | class=pistol_in_holster | confidence=0.717 | iod_det=0.674 (67.4%) | threshold=0.60 | KEEP
00362808-ac72-11f1-8c55-0242ac110014_visual-weapon-detection-frame.jpg | TORSO_XYXY: [0, 528, 566, 1215]
00362808-ac72-11f1-8c55-0242ac110014_visual-weapon-detection-frame.jpg | YOLOX_DET[iod_det≥0.60, thr=0.70]: [377, 1122, 522, 1260], score=0.7169396281242371, cls=pistol_in_holster
IOD MATCH | class=civilian | confidence=0.885 | iod_det

,image_name,guid,predicted_yolox_class,bbox_xyxy,confidence,torso_bbox_xyxy,iod,vwd_prediction
0,0014a0ba-ac73-11f1-8c55-0242ac110014_visual-we...,0014a0ba-ac73-11f1-8c55-0242ac110014,pistol_in_hand,[403 835 483 1049],0.7966657280921936,[38 467 509 1066],1.0,weapon
1,00362808-ac72-11f1-8c55-0242ac110014_visual-we...,00362808-ac72-11f1-8c55-0242ac110014,pistol_in_holster,[377 1122 522 1260],0.7169396281242371,[0 528 566 1215],0.6739130434782609,weapon
2,01059d3c-ac94-11f1-a4e7-0242ac110011_visual-we...,01059d3c-ac94-11f1-a4e7-0242ac110011,"civilian,pistol_in_hand","[127 461 510 945],[382 900 470 1074]","0.8852955102920532,0.7337161302566528",[38 496 579 1119],"0.9276859504132231,1.0",weapon
3,01212740-ac8b-11f1-97d3-0242ac110007_visual-we...,01212740-ac8b-11f1-97d3-0242ac110007,"civilian,pistol_in_holster","[133 432 371 727],[129 572 179 690]","0.8638356924057007,0.7703254818916321",[54 446 395 827],"0.9525423728813559,1.0",weapon
4,0137b64c-ac6b-11f1-bc06-0242ac11000b_visual-we...,0137b64c-ac6b-11f1-bc06-0242ac11000b,pistol_in_hand,[25 487 58 535],0.7352343797683716,[36 447 542 1065],0.6666666666666666,weapon
